# Flare Observations — Implementation / 플레어 관측 구현
# Arnold O. Benz (2008)

This notebook implements key physics concepts from Benz's LRSP review on solar flare observations.
We reproduce core calculations and visualizations covering:

이 노트북은 Benz의 LRSP 태양 플레어 관측 리뷰에서 핵심 물리 개념을 구현합니다:

1. **Neupert Effect** — SXR vs cumulative HXR relationship / SXR과 누적 HXR의 관계
2. **Bremsstrahlung Spectrum** — Thin vs thick target models / Thin vs thick target 모델
3. **Soft-Hard-Soft Behavior** — Spectral index evolution / 스펙트럼 지수 진화
4. **Flare Energy Budget** — Energy partition calculation / 에너지 배분 계산
5. **Stochastic Acceleration (Fokker–Planck)** — Transit-Time Damping / 확률적 가속
6. **Type III Radio Burst Trajectory** — Plasma frequency and coronal density / 플라즈마 주파수와 코로나 밀도
7. **Chromospheric Evaporation** — Pressure-driven upflow model / 압력 구동 상승류 모델

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.integrate import cumulative_trapezoid, solve_ivp
from scipy.constants import k as k_B, m_e, e, pi, c

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

---
## Part 1: Neupert Effect / Neupert 효과

The Neupert effect (Neupert, 1968) states that soft X-ray flux is proportional to
the cumulative hard X-ray flux:

Neupert 효과(Neupert, 1968)는 soft X-ray flux가 누적 hard X-ray flux에 비례한다는 것입니다:

$$F_{SXR}(t) \propto \int_{t_0}^{t} F_{HXR}(t') \, dt'$$

**Physical interpretation / 물리적 해석**: HXR traces the instantaneous energy input rate
by non-thermal electrons, while SXR traces the accumulated thermal energy in the
heated plasma.

HXR은 비열적 전자에 의한 순간 에너지 입력률을, SXR은 가열된 플라즈마의 누적 열 에너지를 추적합니다.

In [ ]:
def synthetic_hxr_lightcurve(t, peaks, widths, amplitudes):
    """Generate a synthetic hard X-ray lightcurve with multiple impulsive bursts.

    Args:
        t: Time array in seconds.
        peaks: List of peak times.
        widths: List of burst widths (sigma).
        amplitudes: List of burst amplitudes.

    Returns:
        HXR flux array.
    """
    flux = np.zeros_like(t)
    for p, w, a in zip(peaks, widths, amplitudes):
        flux += a * np.exp(-0.5 * ((t - p) / w) ** 2)
    return flux


# Simulate a flare with multiple impulsive bursts
t = np.linspace(0, 600, 2000)  # 10 minutes
hxr_peaks = [120, 180, 220, 280, 350]
hxr_widths = [15, 10, 8, 12, 20]
hxr_amps = [3.0, 5.0, 4.5, 6.0, 2.0]

hxr_flux = synthetic_hxr_lightcurve(t, hxr_peaks, hxr_widths, hxr_amps)

# Neupert effect: SXR = cumulative integral of HXR
sxr_flux = np.concatenate([[0], cumulative_trapezoid(hxr_flux, t)])
sxr_flux = sxr_flux / sxr_flux.max()  # Normalize

# Add cooling (exponential decay) for realism
tau_cool = 400  # Cooling time in seconds
sxr_with_cooling = np.zeros_like(t)
dt = t[1] - t[0]
for i in range(1, len(t)):
    sxr_with_cooling[i] = (
        sxr_with_cooling[i - 1] * np.exp(-dt / tau_cool) + hxr_flux[i] * dt
    )
sxr_with_cooling /= sxr_with_cooling.max()

# SXR derivative should match HXR
sxr_deriv = np.gradient(sxr_with_cooling, t)
sxr_deriv = sxr_deriv / sxr_deriv.max() * hxr_flux.max()

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(t, hxr_flux, 'b-', lw=1.5, label='HXR (25-50 keV)')
axes[0].set_ylabel('HXR Flux (arb.)')
axes[0].legend()
axes[0].set_title('Neupert Effect Demonstration / Neupert 효과 시연')

axes[1].plot(t, sxr_flux, 'r-', lw=2, alpha=0.5, label='SXR (no cooling)')
axes[1].plot(t, sxr_with_cooling, 'r-', lw=2, label='SXR (with cooling)')
axes[1].set_ylabel('SXR Flux (arb.)')
axes[1].legend()

axes[2].plot(t, hxr_flux / hxr_flux.max(), 'b-', lw=1.5, alpha=0.7,
             label='HXR (normalized)')
axes[2].plot(t, sxr_deriv / sxr_deriv.max(), 'r--', lw=2,
             label='d(SXR)/dt (normalized)')
axes[2].set_ylabel('Normalized Flux')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## Part 2: Bremsstrahlung Spectrum — Thin vs Thick Target / 제동복사 스펙트럼

For an electron distribution with power-law index $\delta$ ($F(\varepsilon) \propto \varepsilon^{-\delta}$):

전자 분포가 power-law index $\delta$를 가질 때:

- **Thin target**: photon spectrum index $\gamma_{\text{thin}} = \delta + 1$ (steeper / 더 가파름)
- **Thick target**: photon spectrum index $\gamma_{\text{thick}} = \delta - 1$ (flatter / 더 평평함)

The thick target spectrum is flatter because it includes emission from all energies
as the electron decelerates to thermal speed.

Thick target 스펙트럼이 더 평평한 이유는 전자가 열적 속도로 감속될 때까지의
모든 에너지에서의 방출을 포함하기 때문입니다.

In [ ]:
def bremsstrahlung_spectrum(photon_energy, delta, model='thick',
                            e_min=20.0, e_max=500.0):
    """Compute bremsstrahlung photon spectrum from power-law electrons.

    Args:
        photon_energy: Photon energy array in keV.
        delta: Electron spectral index.
        model: 'thin' or 'thick' target.
        e_min: Minimum electron energy in keV.
        e_max: Maximum electron energy in keV.

    Returns:
        Photon flux (arbitrary units).
    """
    if model == 'thin':
        gamma = delta + 1
    else:  # thick
        gamma = delta - 1

    flux = photon_energy ** (-gamma)
    # Apply cutoffs
    flux[photon_energy < e_min * 0.5] *= np.exp(
        -(e_min * 0.5 / photon_energy[photon_energy < e_min * 0.5]) ** 2
    )
    flux[photon_energy > e_max] = 0
    return flux


def thermal_spectrum(photon_energy, temperature_mk):
    """Compute thermal bremsstrahlung spectrum.

    Args:
        photon_energy: Photon energy array in keV.
        temperature_mk: Temperature in MK.

    Returns:
        Photon flux (arbitrary units).
    """
    kT = temperature_mk * 1e6 * k_B / (1.602e-16)  # Convert MK to keV
    return np.exp(-photon_energy / kT) / np.sqrt(photon_energy)


photon_e = np.logspace(0.3, 2.5, 500)  # 2 to ~300 keV
delta_values = [3, 4, 5, 7]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for delta in delta_values:
    thin = bremsstrahlung_spectrum(photon_e, delta, 'thin')
    thick = bremsstrahlung_spectrum(photon_e, delta, 'thick')
    axes[0].loglog(photon_e, thin / thin[100], '--', label=f'Thin, δ={delta}')
    axes[1].loglog(photon_e, thick / thick[100], '-', label=f'Thick, δ={delta}')

# Add thermal component
thermal = thermal_spectrum(photon_e, 16.7)
for ax in axes:
    ax.loglog(photon_e, thermal / thermal[50] * 0.1, 'g-', lw=2, alpha=0.5,
              label='Thermal (16.7 MK)')
    ax.set_xlabel('Photon Energy (keV)')
    ax.set_ylabel('Photon Flux (arb.)')
    ax.set_ylim(1e-8, 1e2)
    ax.legend(fontsize=9)

axes[0].set_title('Thin Target: γ = δ + 1')
axes[1].set_title('Thick Target: γ = δ - 1')

fig.suptitle('Bremsstrahlung Photon Spectra / 제동복사 광자 스펙트럼', fontsize=14,
             y=1.02)
plt.tight_layout()
plt.show()

---
## Part 3: Soft-Hard-Soft Behavior / Soft-Hard-Soft 행동

Grigis & Benz (2004) quantified the soft-hard-soft behavior:

$$\gamma = A \cdot F(E_0)^{-\alpha}$$

where $\gamma$ is the photon spectral index, $F(E_0)$ is the flux density at $E_0 = 35$ keV,
and $\alpha \approx 0.12$ (rise phase), $\alpha \approx 0.17$ (decay phase).

Higher flux → harder (flatter) spectrum → more efficient acceleration.

높은 flux → 더 hard한(flat한) 스펙트럼 → 더 효율적인 가속.

In [ ]:
def simulate_shs_flare(t, t_peak=200, rise_time=60, decay_time=120):
    """Simulate a flare lightcurve and its soft-hard-soft spectral evolution.

    Args:
        t: Time array.
        t_peak: Peak time.
        rise_time: Rise timescale.
        decay_time: Decay timescale.

    Returns:
        flux: Flux at 35 keV.
        gamma: Photon spectral index.
    """
    # Asymmetric pulse
    flux = np.where(
        t < t_peak,
        np.exp(-0.5 * ((t - t_peak) / rise_time) ** 2),
        np.exp(-0.5 * ((t - t_peak) / decay_time) ** 2)
    )
    flux = flux * 10 + 0.01  # Scale and add background

    # Soft-hard-soft relation
    A_rise = 8.0
    A_decay = 7.5
    alpha_rise = 0.121
    alpha_decay = 0.172

    gamma = np.where(
        t < t_peak,
        A_rise * flux ** (-alpha_rise),
        A_decay * flux ** (-alpha_decay)
    )
    return flux, gamma


t_shs = np.linspace(0, 500, 1000)
flux_35, gamma_spec = simulate_shs_flare(t_shs)

fig = plt.figure(figsize=(14, 5))
gs = GridSpec(1, 3, figure=fig, width_ratios=[1, 1, 1])

# Lightcurve
ax1 = fig.add_subplot(gs[0])
ax1.semilogy(t_shs, flux_35, 'b-', lw=2)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Flux at 35 keV (arb.)')
ax1.set_title('HXR Lightcurve')
ax1.axvline(200, color='gray', ls='--', alpha=0.5, label='Peak')
ax1.legend()

# Spectral index evolution
ax2 = fig.add_subplot(gs[1])
colors = plt.cm.coolwarm(np.linspace(0, 1, len(t_shs)))
for i in range(len(t_shs) - 1):
    ax2.plot(t_shs[i:i+2], gamma_spec[i:i+2], color=colors[i], lw=2)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Photon Spectral Index γ')
ax2.set_title('Spectral Evolution')
ax2.invert_yaxis()  # Lower gamma = harder
ax2.axvline(200, color='gray', ls='--', alpha=0.5)
ax2.annotate('HARD', xy=(200, gamma_spec.min() + 0.1), fontsize=10,
             ha='center', color='red')
ax2.annotate('SOFT', xy=(50, gamma_spec[50] - 0.1), fontsize=10,
             ha='center', color='blue')

# Flux vs gamma (the SHS relation)
ax3 = fig.add_subplot(gs[2])
mask_rise = t_shs < 200
mask_decay = t_shs >= 200
ax3.plot(flux_35[mask_rise], gamma_spec[mask_rise], 'b-', lw=2,
         label=f'Rise (α={0.121})')
ax3.plot(flux_35[mask_decay], gamma_spec[mask_decay], 'r-', lw=2,
         label=f'Decay (α={0.172})')
ax3.set_xscale('log')
ax3.set_xlabel('Flux at 35 keV')
ax3.set_ylabel('γ (Spectral Index)')
ax3.set_title('γ = A·F⁻ᵅ (Grigis & Benz 2004)')
ax3.invert_yaxis()
ax3.legend()

fig.suptitle('Soft-Hard-Soft Behavior / Soft-Hard-Soft 행동', fontsize=14,
             y=1.02)
plt.tight_layout()
plt.show()

---
## Part 4: Flare Energy Budget / 플레어 에너지 수지

We calculate the energy partition for flares of different GOES classes,
based on Table 1 of Benz (2008).

Benz (2008)의 Table 1에 기반하여 다양한 GOES 등급 플레어의 에너지 배분을 계산합니다.

Key equations:
- Non-thermal electron energy: $E_{\text{kin}} = \int F(\varepsilon)\varepsilon \, d\varepsilon$
- Thermal energy: $E_{\text{th}} = 3 k_B T \sqrt{MV}$

In [ ]:
def nonthermal_energy(delta, e_min_kev, e_max_kev, flux_norm, duration_s):
    """Calculate non-thermal electron energy from power-law distribution.

    Args:
        delta: Electron spectral index.
        e_min_kev: Low-energy cutoff in keV.
        e_max_kev: High-energy cutoff in keV.
        flux_norm: Normalization of electron flux (electrons/s/keV at 1 keV).
        duration_s: Duration of impulsive phase in seconds.

    Returns:
        Total non-thermal energy in erg.
    """
    kev_to_erg = 1.602e-9  # 1 keV in erg
    # Integral of e^(2-delta) from e_min to e_max
    if delta != 2:
        integral = (e_max_kev ** (2 - delta) - e_min_kev ** (2 - delta)) / (
            2 - delta
        )
    else:
        integral = np.log(e_max_kev / e_min_kev)
    return flux_norm * integral * kev_to_erg * duration_s


def thermal_energy(T_mk, em_cm3, volume_cm3):
    """Calculate thermal energy from emission measure.

    Args:
        T_mk: Temperature in MK.
        em_cm3: Emission measure in cm^-3.
        volume_cm3: Source volume in cm^3.

    Returns:
        Thermal energy in erg.
    """
    k_cgs = 1.381e-16  # Boltzmann constant in erg/K
    T_k = T_mk * 1e6
    return 3 * k_cgs * T_k * np.sqrt(em_cm3 * volume_cm3)


# Energy budget comparison (based on Table 1)
flare_classes = ['M2.6\n2002/11/10', 'M7.8\n2002/08/22',
                 'X1.7\n2002/04/21', 'X5.1\n2002/07/23']
e_nonthermal = [1.9e30, 6.5e30, 2.0e31, 3.2e31]  # erg
e_thermal = [1.4e30, 2.6e30, 1.2e31, 1.0e31]  # erg
e_radiated = [np.nan, np.nan, 1.6e32, 1.6e32]  # erg
e_cme_kin = [np.nan, np.nan, 2.0e32, 1.0e32]  # erg

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart of energy components
x = np.arange(len(flare_classes))
width = 0.3
axes[0].bar(x - width / 2, e_nonthermal, width, label='Non-thermal electrons',
            color='royalblue', alpha=0.8)
axes[0].bar(x + width / 2, e_thermal, width, label='Thermal plasma',
            color='orangered', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(flare_classes, fontsize=9)
axes[0].set_yscale('log')
axes[0].set_ylabel('Energy (erg)')
axes[0].set_title('Non-thermal vs Thermal Energy')
axes[0].legend()

# Ratio plot
ratio = np.array(e_nonthermal) / np.array(e_thermal)
colors = ['green' if r > 1 else 'red' for r in ratio]
axes[1].bar(x, ratio, 0.5, color=colors, alpha=0.8)
axes[1].axhline(1, color='k', ls='--', lw=1)
axes[1].set_xticks(x)
axes[1].set_xticklabels(flare_classes, fontsize=9)
axes[1].set_ylabel('E_kin / E_th')
axes[1].set_title('Energy Ratio: Non-thermal / Thermal')
axes[1].annotate('E_kin > E_th in ALL cases!',
                 xy=(1.5, max(ratio) * 0.9), fontsize=11,
                 ha='center', color='darkgreen', fontweight='bold')

fig.suptitle('Flare Energy Budget (Benz 2008, Table 1) / 플레어 에너지 수지',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Energy partition pie chart for X-class flare
fig, ax = plt.subplots(figsize=(8, 8))
labels = ['White Light + IR\n(77%)', 'UV + SXR\n(23%)']
sizes = [77, 23]
colors_pie = ['#FFD700', '#4169E1']
ax.pie(sizes, labels=labels, colors=colors_pie, autopct='%1.0f%%',
       startangle=90, textprops={'fontsize': 14})
ax.set_title('Total Flare Irradiance Partition / 총 플레어 복사조도 배분',
             fontsize=14)
plt.show()

---
## Part 5: Stochastic Acceleration — Fokker-Planck / 확률적 가속

The Fokker-Planck equation for stochastic acceleration by Transit-Time Damping:

$$\frac{\partial f(p)}{\partial t} = \frac{1}{p^2} \frac{\partial}{\partial p} \left[ p^2 \left( D(p) \frac{\partial f}{\partial p} + F(p) f \right) \right]$$

where $D(p)$ is the momentum diffusion coefficient (from waves) and $F(p)$
represents Coulomb friction.

We solve a simplified 1D version in energy space.

파동에 의한 확산과 Coulomb 충돌에 의한 감속의 경쟁을 1D 에너지 공간에서 간소화하여 풀겠습니다.

In [ ]:
def solve_fokker_planck_1d(n_energy=200, n_time=500, e_max_kev=200,
                           t_max_s=2.0, D0=5.0, T_plasma_mk=2.0,
                           n_e_cm3=1e10):
    """Solve simplified 1D Fokker-Planck equation for electron acceleration.

    Models stochastic acceleration (diffusion in energy space) competing
    with Coulomb collisions (friction).

    Args:
        n_energy: Number of energy grid points.
        n_time: Number of time steps.
        e_max_kev: Maximum energy in keV.
        t_max_s: Maximum time in seconds.
        D0: Diffusion coefficient strength.
        T_plasma_mk: Plasma temperature in MK.
        n_e_cm3: Electron density in cm^-3.

    Returns:
        energy: Energy grid in keV.
        times: Time array.
        f_history: Distribution function at selected times.
    """
    energy = np.linspace(0.5, e_max_kev, n_energy)
    de = energy[1] - energy[0]
    dt = t_max_s / n_time

    # Initial Maxwellian distribution
    kT = T_plasma_mk * 1e6 * k_B / 1.602e-16  # in keV
    f = np.exp(-energy / kT) * energy ** 0.5  # Maxwellian
    f = f / f.sum()  # Normalize

    # Diffusion coefficient: D(E) ~ D0 * E (energy-dependent)
    D = D0 * energy

    # Coulomb friction: F(E) ~ -C / E (stronger at low energy)
    coulomb_log = 20
    C_friction = 0.5 * n_e_cm3 / 1e10 * coulomb_log
    F_friction = C_friction / (energy + 0.1)

    # Escape term: particles escape when acceleration balances escape
    tau_escape = 1.0  # seconds
    escape_rate = 1.0 / tau_escape * (energy > 10)  # Only high-E escape

    snapshot_times = [0, 0.1, 0.3, 0.5, 1.0, 2.0]
    f_history = {0: f.copy()}

    for step in range(1, n_time + 1):
        t_current = step * dt

        # Finite difference for diffusion term
        d2f = np.zeros_like(f)
        d2f[1:-1] = (
            D[1:-1] * (f[2:] - 2 * f[1:-1] + f[:-2]) / de ** 2
            + (D[2:] - D[:-2]) / (2 * de) * (f[2:] - f[:-2]) / (2 * de)
        )

        # Friction term
        df_friction = np.zeros_like(f)
        df_friction[1:-1] = (
            F_friction[1:-1] * (f[2:] - f[:-2]) / (2 * de)
            + f[1:-1] * (F_friction[2:] - F_friction[:-2]) / (2 * de)
        )

        # Update
        f = f + dt * (d2f + df_friction - escape_rate * f)
        f = np.maximum(f, 1e-30)  # Prevent negatives

        # Boundary conditions
        f[0] = f[1]
        f[-1] = 0

        for st in snapshot_times:
            if abs(t_current - st) < dt / 2 and st not in f_history:
                f_history[st] = f.copy()

    return energy, snapshot_times, f_history


energy, times, f_history = solve_fokker_planck_1d()

fig, ax = plt.subplots(figsize=(10, 7))
colors_fp = plt.cm.viridis(np.linspace(0, 1, len(times)))

for i, t_snap in enumerate(sorted(f_history.keys())):
    f_snap = f_history[t_snap]
    ax.semilogy(energy, f_snap, color=colors_fp[i], lw=2,
                label=f't = {t_snap:.1f} s')

# Overlay power-law fits for comparison
for delta in [3.5, 5, 7]:
    mask = (energy > 15) & (energy < 100)
    pl = 1e-4 * energy[mask] ** (-delta)
    ax.semilogy(energy[mask], pl, 'k--', alpha=0.3, lw=1)
    ax.text(energy[mask][-1] * 1.05, pl[-1], f'δ={delta}', fontsize=8,
            alpha=0.5)

ax.set_xlabel('Energy (keV)')
ax.set_ylabel('f(E) (arb.)')
ax.set_title('Fokker-Planck: Stochastic Acceleration (TTD)\n'
             'Non-thermal tail develops from Maxwellian / '
             'Maxwellian에서 비열적 꼬리 발달')
ax.set_ylim(1e-15, 1e0)
ax.set_xlim(0, 150)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## Part 6: Type III Radio Burst Trajectory / Type III 전파 버스트 궤적

Type III radio bursts are emitted at the local plasma frequency:

$$\omega_p = \sqrt{\frac{4\pi e^2 n_e}{m_e}} \quad \Rightarrow \quad f_p \approx 9 \sqrt{n_e} \text{ Hz}$$

As an electron beam propagates outward through the corona, the decreasing
density causes a characteristic frequency drift from high to low frequencies.

전자빔이 코로나를 따라 바깥으로 전파하면, 밀도 감소로 인해 고주파에서 저주파로의
특징적 주파수 드리프트가 발생합니다.

In [ ]:
def coronal_density_model(height_rsun, model='newkirk'):
    """Coronal electron density as a function of height.

    Args:
        height_rsun: Height in solar radii (1 = photosphere).
        model: Density model ('newkirk' or 'barometric').

    Returns:
        Electron density in cm^-3.
    """
    if model == 'newkirk':
        # Newkirk (1961) model: n = n0 * 10^(4.32/r)
        n0 = 4.2e4  # cm^-3
        return n0 * 10 ** (4.32 / height_rsun)
    else:
        # Barometric (exponential)
        R_sun = 6.96e10  # cm
        H = 5e9  # Scale height cm
        n0 = 1e9
        return n0 * np.exp(-(height_rsun - 1) * R_sun / H)


def plasma_frequency(n_e_cm3):
    """Calculate plasma frequency in MHz from electron density.

    Args:
        n_e_cm3: Electron density in cm^-3.

    Returns:
        Plasma frequency in MHz.
    """
    return 9e-3 * np.sqrt(n_e_cm3)  # in MHz


# Simulate electron beam propagating through corona
heights = np.linspace(1.05, 2.0, 500)  # In solar radii
ne_newkirk = coronal_density_model(heights, 'newkirk')
fp_newkirk = plasma_frequency(ne_newkirk)
fp_harmonic = 2 * fp_newkirk  # Second harmonic

# Electron beam speed ~ 0.3c
R_sun = 6.96e10  # cm
v_beam = 0.3 * 3e10  # cm/s
travel_time = (heights - heights[0]) * R_sun / v_beam  # seconds

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Density profile
axes[0].semilogy(heights, ne_newkirk, 'b-', lw=2)
axes[0].set_xlabel('Height (R☉)')
axes[0].set_ylabel('n_e (cm⁻³)')
axes[0].set_title('Coronal Density (Newkirk)')

# Dynamic spectrum (frequency vs time)
axes[1].plot(travel_time, fp_newkirk, 'b-', lw=2, label='Fundamental')
axes[1].plot(travel_time, fp_harmonic, 'r--', lw=2, label='2nd Harmonic')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Frequency (MHz)')
axes[1].set_title('Type III Frequency Drift')
axes[1].invert_yaxis()
axes[1].legend()

# Simulated dynamic spectrum (spectrogram)
t_grid = np.linspace(0, travel_time[-1], 200)
f_grid = np.linspace(fp_newkirk.min(), fp_newkirk.max(), 200)
T_grid, F_grid = np.meshgrid(t_grid, f_grid)

# Create a narrow emission band following the drift
intensity = np.zeros_like(T_grid)
for i, (tt, ff) in enumerate(zip(travel_time[::5], fp_newkirk[::5])):
    intensity += np.exp(-((T_grid - tt) / 0.02) ** 2
                        - ((F_grid - ff) / 10) ** 2)

axes[2].pcolormesh(t_grid, f_grid, intensity, cmap='hot', shading='auto')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Frequency (MHz)')
axes[2].set_title('Simulated Type III Spectrogram')
axes[2].invert_yaxis()

fig.suptitle('Type III Radio Burst / Type III 전파 버스트', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 7: Chromospheric Evaporation / 채층 증발

When non-thermal electrons deposit energy in the chromosphere, the heated plasma
expands along the magnetic field into the corona. We model this as a
pressure-driven flow:

비열적 전자가 채층에 에너지를 전달하면, 가열된 플라즈마가 자기장을 따라 코로나로
팽창합니다. 이를 압력 구동 유동으로 모델링합니다:

- **Explosive evaporation**: $F_{\text{beam}} > 3 \times 10^{10}$ erg cm$^{-2}$ s$^{-1}$ → supersonic upflow (300-400 km/s)
- **Gentle evaporation**: $F_{\text{beam}} < 3 \times 10^{10}$ erg cm$^{-2}$ s$^{-1}$ → subsonic upflow (~65 km/s)

In [ ]:
def chromospheric_evaporation_model(beam_flux_cgs, loop_length_cm=2e9,
                                     n_chrom_cm3=1e12, T_chrom_k=1e4):
    """Simple model for chromospheric evaporation velocity.

    Based on the balance between beam energy deposition and
    pressure-driven expansion.

    Args:
        beam_flux_cgs: Electron beam energy flux in erg/cm^2/s.
        loop_length_cm: Half-length of the coronal loop in cm.
        n_chrom_cm3: Chromospheric density in cm^-3.
        T_chrom_k: Initial chromospheric temperature in K.

    Returns:
        v_evap: Evaporation velocity in km/s.
        T_heated: Heated temperature in MK.
        regime: 'explosive' or 'gentle'.
    """
    k_cgs = 1.381e-16  # erg/K
    m_p = 1.673e-24  # g

    # Threshold for explosive evaporation
    threshold = 3e10  # erg/cm^2/s

    # Heated temperature (energy balance: beam flux ~ n * k * T * v)
    # Approximate: T ~ (beam_flux / (n * k * c_s))^(2/3)
    c_s = np.sqrt(2 * k_cgs * T_chrom_k / m_p)  # Sound speed
    T_heated = (beam_flux_cgs / (n_chrom_cm3 * k_cgs * c_s)) ** 0.4 * T_chrom_k
    T_heated_mk = T_heated / 1e6

    # Evaporation velocity ~ sound speed at heated temperature
    c_s_heated = np.sqrt(2 * k_cgs * T_heated / m_p)  # cm/s
    v_evap = c_s_heated / 1e5  # Convert to km/s

    # Regime classification
    regime = np.where(beam_flux_cgs > threshold, 'explosive', 'gentle')

    return v_evap, T_heated_mk, regime


# Calculate for a range of beam fluxes
beam_fluxes = np.logspace(8, 12, 200)
v_evap, T_heated, regimes = chromospheric_evaporation_model(beam_fluxes)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Velocity vs beam flux
threshold = 3e10
mask_gentle = beam_fluxes <= threshold
mask_explosive = beam_fluxes > threshold

axes[0].semilogx(beam_fluxes[mask_gentle], v_evap[mask_gentle], 'b-', lw=2,
                 label='Gentle evaporation')
axes[0].semilogx(beam_fluxes[mask_explosive], v_evap[mask_explosive], 'r-',
                 lw=2, label='Explosive evaporation')
axes[0].axvline(threshold, color='gray', ls='--', lw=1.5,
                label=f'Threshold: 3×10¹⁰')
axes[0].axhline(65, color='blue', ls=':', alpha=0.5, label='~65 km/s (gentle)')
axes[0].axhline(350, color='red', ls=':', alpha=0.5, label='~350 km/s (explosive)')
axes[0].set_xlabel('Beam Energy Flux (erg cm⁻² s⁻¹)')
axes[0].set_ylabel('Evaporation Velocity (km/s)')
axes[0].set_title('Evaporation Velocity vs Beam Flux')
axes[0].legend(fontsize=9)

# Temperature vs beam flux
axes[1].loglog(beam_fluxes[mask_gentle], T_heated[mask_gentle], 'b-', lw=2,
               label='Gentle')
axes[1].loglog(beam_fluxes[mask_explosive], T_heated[mask_explosive], 'r-',
               lw=2, label='Explosive')
axes[1].axvline(threshold, color='gray', ls='--', lw=1.5)
axes[1].axhline(20, color='orange', ls=':', alpha=0.5,
                label='~20 MK (observed Ca XIX)')
axes[1].set_xlabel('Beam Energy Flux (erg cm⁻² s⁻¹)')
axes[1].set_ylabel('Heated Temperature (MK)')
axes[1].set_title('Heated Temperature vs Beam Flux')
axes[1].legend(fontsize=9)

fig.suptitle('Chromospheric Evaporation Model / 채층 증발 모델', fontsize=14,
             y=1.02)
plt.tight_layout()
plt.show()

---
## Summary / 요약

| Part | Topic / 주제 | Key Result / 핵심 결과 |
|---|---|---|
| 1 | Neupert Effect | $F_{SXR} \propto \int F_{HXR} dt$ — SXR is cumulative HXR |
| 2 | Bremsstrahlung | Thick target: $\gamma = \delta - 1$ (flatter than thin: $\gamma = \delta + 1$) |
| 3 | Soft-Hard-Soft | $\gamma = A \cdot F^{-\alpha}$ — higher flux → harder spectrum |
| 4 | Energy Budget | $E_{kin} > E_{th}$ for all observed flares |
| 5 | Fokker-Planck | Non-thermal power-law tail develops from Maxwellian via TTD |
| 6 | Type III Burst | Frequency drift traces electron beam through corona |
| 7 | Evaporation | Threshold at $3 \times 10^{10}$ erg cm$^{-2}$ s$^{-1}$ separates gentle/explosive |

### Connection to Next Paper / 다음 논문과의 연결

| Paper | Relevance / 관련성 |
|---|---|
| LRSP #14 Hall (2008) | Stellar chromospheric activity — flare physics extended to stellar context / 항성 채층 활동으로 태양 플레어 물리의 확장 |
| LRSP #9 Schwenn (2006) | CME observations — how flare energy couples to interplanetary space / CME 관측과 플레어 에너지의 행성간 공간 결합 |